# 01. ESS 배터리 데이터 점검 및 탐색적 데이터 분석 (EDA)

## 1. 프로젝트 개요 및 원본 데이터 정리

### 데이터 파일 구성 (`data/`)
| 파일명 | 크기 | 식별명 | 셀 수 | 역할 및 적용 범위 |
| :--- | :--- | :--- | :---: | :--- |
| `2017-05-12_batchdata_updated_struct_errorcorrect.mat` | ~2.8 GB | **Batch 1** | 46 | **Train & 내부 Hold-out 검증용** (기본 학습셋) |
| `2018-02-20_batchdata_updated_struct_errorcorrect.mat` | ~1.9 GB | **Batch 2** | 47 | **최종 일반화 Test용** (필수 평가셋) |
| `2018-04-12_batchdata_updated_struct_errorcorrect.mat` | ~3.0 GB | **Batch 3** | 46 | **추가 비교 분석용** (Batch 1·2 완료 후 진행) |
| `2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat` | ~85 MB | **Extra** | - | 충전 최적화 연구 데이터 |

> - **1단계**: Batch 1(학습/검증)과 Batch 2(테스트)를 기준으로 파이프라인(EDA → 피처 엔지니어링 → 모델링)을 완결합니다.
> - **2단계**: 완성된 모델을 Batch 3에 추가 적용하여 배치 간 일반화 성능을 확장 검증합니다.

In [9]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 프로젝트 루트 등록
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.preprocess import load_batch_summary

# 시각화 스타일 설정
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'AppleGothic' if sys.platform == 'darwin' else 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = os.path.join(project_root, 'data')
print(f"Data Directory: {DATA_DIR}")

Data Directory: /Users/skala_yh/DS_miniproject/data


## 2. Batch 1 & Batch 2 데이터 로드

In [10]:
b1_path = os.path.join(DATA_DIR, '2017-05-12_batchdata_updated_struct_errorcorrect.mat')
b2_path = os.path.join(DATA_DIR, '2018-02-20_batchdata_updated_struct_errorcorrect.mat')

df_b1 = load_batch_summary(b1_path, 'Batch1')
df_b2 = load_batch_summary(b2_path, 'Batch2')

print(f"Batch 1 로드 셀 수: {len(df_b1)}")
print(f"Batch 2 로드 셀 수: {len(df_b2)}")

Batch 1 로드 셀 수: 46
Batch 2 로드 셀 수: 47


## 3. 셀 단위 초기 지표 추출 및 기초 통계량 산출
- 배터리 수명(`cycle_life`), 초기 용량(Cycle 2 기준), 100사이클 용량, 초기 저항, 최고 온도, 평균 충전시간을 집계합니다.

In [11]:
def extract_cell_features(df):
    rows = []
    for _, r in df.iterrows():
        qd = r['QDischarge']
        ir = r['IR']
        tmax = r['Tmax']
        ch_time = r['chargetime']
        
        q_init = qd[1] if len(qd) > 1 else np.nan
        q_100 = qd[99] if len(qd) > 99 else np.nan
        ir_init = ir[1] if len(ir) > 1 else np.nan
        ir_100 = ir[99] if len(ir) > 99 else np.nan
        t_max_100 = np.max(tmax[:100]) if len(tmax) >= 100 else np.nan
        ch_time_avg = np.mean(ch_time[1:100]) if len(ch_time) >= 100 else np.nan
        
        rows.append({
            'cell_id': r['cell_id'],
            'batch': r['batch'],
            'policy': r['policy'],
            'cycle_life': r['cycle_life'],
            'n_cycles': r['n_cycles'],
            'q_init_c2': q_init,
            'q_c100': q_100,
            'delta_q_100_2': (q_100 - q_init) if (not np.isnan(q_100) and not np.isnan(q_init)) else np.nan,
            'ir_init': ir_init,
            'ir_c100': ir_100,
            't_max_100': t_max_100,
            'ch_time_avg_min': ch_time_avg
        })
    return pd.DataFrame(rows)

feat_b1 = extract_cell_features(df_b1)
feat_b2 = extract_cell_features(df_b2)
df_feat_all = pd.concat([feat_b1, feat_b2], ignore_index=True)

print("=== [표 1] Batch 1 기초 통계량 ===")
display(feat_b1.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

print("=== [표 2] Batch 2 기초 통계량 ===")
display(feat_b2.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

=== [표 1] Batch 1 기초 통계량 ===


,count,mean,std,min,50%,max
cycle_life,46.0,844.717391,184.629198,534.000000,858.500000,1227.000000
n_cycles,46.0,843.717391,184.629198,533.000000,857.500000,1226.000000
q_init_c2,46.0,1.078083,0.007652,1.053779,1.078689,1.094639
q_c100,46.0,1.080683,0.007750,1.059972,1.081524,1.095762
delta_q_100_2,46.0,0.002600,0.003022,-0.011382,0.002805,0.006626
ir_init,46.0,0.016720,0.000611,0.015349,0.016738,0.019886
ir_c100,46.0,0.016612,0.000443,0.015145,0.016683,0.017589
t_max_100,46.0,36.784144,1.569721,31.658901,37.176383,38.728722
ch_time_avg_min,46.0,11.281064,1.408992,8.987271,11.145064,15.075768


=== [표 2] Batch 2 기초 통계량 ===


,count,mean,std,min,50%,max
cycle_life,39.0,565.743590,222.201629,392.000000,472.000000,1186.000000
n_cycles,47.0,572.425532,258.388210,101.000000,493.000000,1252.000000
q_init_c2,47.0,1.091357,0.010809,1.067174,1.092568,1.109278
q_c100,47.0,1.181702,0.300639,1.003442,1.097549,2.167388
delta_q_100_2,47.0,0.090345,0.303457,-0.068793,0.002183,1.083970
ir_init,47.0,0.014662,0.006357,0.000000,0.016828,0.021771
ir_c100,47.0,0.015005,0.006896,0.000000,0.016515,0.027307
t_max_100,47.0,45.131456,52.923123,31.867259,37.579971,400.000000
ch_time_avg_min,47.0,20.943147,16.871345,10.035778,10.181223,54.544896


## 4. [비교 요약표] Batch 1 vs Batch 2 핵심 통계 비교

In [12]:
comparison_table = df_feat_all.groupby('batch').agg(
    전체_셀수=('cell_id', 'count'),
    유효_수명_셀수=('cycle_life', 'count'),
    수명_결측_셀수=('cycle_life', lambda s: s.isna().sum()),
    수명_평균=('cycle_life', 'mean'),
    수명_표준편차=('cycle_life', 'std'),
    수명_최솟값=('cycle_life', 'min'),
    수명_중위수=('cycle_life', 'median'),
    수명_최댓값=('cycle_life', 'max'),
    초기용량_평균=('q_init_c2', 'mean'),
    초기내부저항_평균=('ir_init', 'mean'),
    온도_이상치_셀수=('t_max_100', lambda s: (s > 100).sum())
).T

display(comparison_table)

batch,Batch1,Batch2
전체_셀수,46.000000,47.000000
유효_수명_셀수,46.000000,39.000000
수명_결측_셀수,0.000000,8.000000
수명_평균,844.717391,565.743590
수명_표준편차,184.629198,222.201629
수명_최솟값,534.000000,392.000000
수명_중위수,858.500000,472.000000
수명_최댓값,1227.000000,1186.000000
초기용량_평균,1.078083,1.091357
초기내부저항_평균,0.016720,0.014662


## 5. 기초 통계량 분석 및 주요 발견 (Insights)

위 통계량 표를 통해 다음과 같은 핵심 사실들을 발견할 수 있습니다:

1. **Batch 2의 타깃 결측 셀 8개 발견**:
   - Batch 1은 46개 셀 모두 정상적인 `cycle_life` 라벨을 가짐 (결측 0개).
   - Batch 2는 47개 중 **8개 셀이 `cycle_life = NaN`**임 (`VarCharge` 4개, `SLOWCYCLE` 4개).
   - **조치**: 이 8개 셀은 EOL(80% 용량 저하)에 도달하지 않은 특수 실험이므로 모델 평가 대상에서 제외해야 함 (원논문 기준 유효 평가 셀 = 39개).

2. **배치 간 수명 분포의 뚜렷한 차이 (Distribution Shift)**:
   - **Batch 1**: 평균 수명 **844.7 사이클** (최소 534 ~ 최대 1,227 사이클).
   - **Batch 2**: 평균 수명 **565.7 사이클** (최소 392 ~ 최대 1,186 사이클, 중위수 472 사이클).
   - Batch 2의 수명이 전반적으로 훨씬 짧음. 이는 단순 무작위 분할(Random split)이 아닌 **배치 간 일반화(Batch-to-batch generalization)**가 본 과제의 핵심 도전과제임을 시사함.

3. **센서 이상치(Glitch) 및 노이즈 발견**:
   - Batch 2에서 `t_max_100`의 최댓값이 **400°C**로 기록된 셀 존재 (배터리 작동 범위 정상치 ~30-40°C를 벗어난 센서 오류).
   - 또한 Cycle 100 용량이 2.16 Ah 이상으로 튀는 셀 존재 (공칭 1.1 Ah).
   - **조치**: 피처 엔지니어링 시 센서 글리치 클리핑 또는 이상 셀 정제 로직 필수.